In [1]:
# =====================================================
# ASL SIGN LANGUAGE TRAINING (ONE CELL COLAB)
# =====================================================

# Install
!pip -q install kaggle

# Import
import os
import zipfile
import numpy as np
import tensorflow as tf
from google.colab import files
from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras.applications import MobileNetV2
import matplotlib.pyplot as plt

# =====================================================
# UPLOAD KAGGLE.JSON
# =====================================================

print("UPLOAD FILE kaggle.json")
uploaded = files.upload()

os.makedirs("/root/.kaggle", exist_ok=True)

for filename in uploaded.keys():
    os.rename(filename, "/root/.kaggle/kaggle.json")

os.chmod("/root/.kaggle/kaggle.json", 0o600)

# =====================================================
# DOWNLOAD DATASET
# =====================================================

print("Downloading ASL Dataset...")

!kaggle datasets download grassknoted/asl-alphabet -q

print("Extracting Dataset...")

with zipfile.ZipFile("asl-alphabet.zip", "r") as zip_ref:
    zip_ref.extractall("/content")

# =====================================================
# DATASET PATH
# =====================================================

DATASET_PATH = "/content/asl_alphabet_train"

IMG_SIZE = (224,224)
BATCH_SIZE = 32
SEED = 42

# =====================================================
# LOAD DATASET
# =====================================================

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names

print("\nJumlah Kelas:", len(class_names))
print(class_names)

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

# =====================================================
# DATA AUGMENTATION
# =====================================================

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# =====================================================
# MOBILENETV2
# =====================================================

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

# =====================================================
# BUILD MODEL
# =====================================================

model = models.Sequential([
    data_augmentation,

    layers.Rescaling(1./255),

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dropout(0.3),

    layers.Dense(
        256,
        activation="relu"
    ),

    layers.Dense(
        len(class_names),
        activation="softmax"
    )
])

# =====================================================
# COMPILE
# =====================================================

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# =====================================================
# CALLBACK
# =====================================================

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "best_model.keras",
    monitor="val_accuracy",
    save_best_only=True
)

# =====================================================
# TRAIN
# =====================================================

EPOCHS = 10

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[
        early_stop,
        checkpoint
    ]
)

# =====================================================
# EVALUATE
# =====================================================

loss, accuracy = model.evaluate(val_ds)

print("\n==============================")
print("FINAL ACCURACY:", accuracy)
print("==============================")

# =====================================================
# SAVE MODEL
# =====================================================

model.save("sign_language_model.keras")

print("\nMODEL SAVED")

# =====================================================
# PLOT
# =====================================================

plt.figure(figsize=(10,5))
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title("Training Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend(["Train","Validation"])
plt.show()

# =====================================================
# DOWNLOAD MODEL
# =====================================================

print("\nDownloading model...")

files.download("sign_language_model.keras")


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


ModuleNotFoundError: No module named 'google.colab'